In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import models
from sklearn.utils.class_weight import compute_class_weight

In [2]:
BASE_DIR = Path(os.getcwd()).parent
sys.path.append(str(BASE_DIR))

from api.app.v1.flowers.models import FlowerDataset, SubsetWithTransform
from api.app.v1.flowers.training_utils import (
    get_device,
    get_transforms,
    split_dataset,
    train,
    val_epoch,
)

In [3]:
device = get_device()
data_dir = BASE_DIR / "data"

Using device: cuda


In [4]:
# init fine tuning model
model = models.efficientnet_b0(weights="IMAGENET1K_V1")
# get transforms
train_transform, val_transform = get_transforms()
# get oxford flowers dataset
dataset = FlowerDataset(data_dir)
# get subsets
train_subset, val_subset, test_subset = split_dataset(dataset, 0.7, 0.15, 0.15)

# apply transforms
train_subset = SubsetWithTransform(train_subset, train_transform)
val_subset = SubsetWithTransform(val_subset, val_transform)
test_subset = SubsetWithTransform(test_subset, val_transform)

In [5]:
# get loaders
batch_size = 64
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

In [6]:
# compute class weight distribution for uneven dataset
labels = dataset.labels
classes = np.unique(labels)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels)
# convert to tensor
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

In [7]:
# view model layout
print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [8]:
print(type(model.classifier[1].in_features))

<class 'int'>


In [10]:
# start by freezing the backbone, replace the final FC layer with a 102-class head
for param in model.parameters():
    param.requires_grad = False

old_classifier = model.classifier

new_classifier_lin_lay = nn.Linear(old_classifier[1].in_features, len(classes))  # type: ignore

model.classifier[1] = new_classifier_lin_lay
for param in model.classifier.parameters():
    param.requires_grad = True

In [11]:
loss_function = nn.CrossEntropyLoss(weight=class_weights)  # type: ignore
optimizer = optim.Adam(model.parameters(), weight_decay=0.0005, lr=1e-4)

In [13]:
num_epochs = 20
model, metrics = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_function=loss_function,
    optimizer=optimizer,
    scheduler=None,
    num_epochs=num_epochs,
    device=device,
)

--- Training Started ---
Epoch [1/20], Train Loss: 285.1814, Val Loss: 263.1570, Val Accuracy: 23.62%
Epoch [2/20], Train Loss: 261.6699, Val Loss: 244.4354, Val Accuracy: 46.99%
Epoch [3/20], Train Loss: 240.8930, Val Loss: 226.9689, Val Accuracy: 58.31%
Epoch [4/20], Train Loss: 221.8093, Val Loss: 210.4054, Val Accuracy: 64.25%
Epoch [5/20], Train Loss: 204.0985, Val Loss: 195.9298, Val Accuracy: 66.61%
Epoch [6/20], Train Loss: 188.1294, Val Loss: 183.7463, Val Accuracy: 68.89%
Epoch [7/20], Train Loss: 173.1651, Val Loss: 170.8298, Val Accuracy: 71.42%
Epoch [8/20], Train Loss: 160.9978, Val Loss: 159.2081, Val Accuracy: 73.13%
Epoch [9/20], Train Loss: 149.4308, Val Loss: 150.8291, Val Accuracy: 73.62%
Epoch [10/20], Train Loss: 138.7987, Val Loss: 142.4521, Val Accuracy: 74.76%
Epoch [11/20], Train Loss: 129.2537, Val Loss: 133.1323, Val Accuracy: 75.98%
Epoch [12/20], Train Loss: 121.9805, Val Loss: 127.3594, Val Accuracy: 75.73%
Epoch [13/20], Train Loss: 114.9940, Val Loss: 1

In [29]:
def partial_freeze(model: models.EfficientNet, layers_to_unfreeze: int) -> None:
    # freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    conv_layers = model.features

    for i in range(layers_to_unfreeze):
        layer_to_unfreeze = conv_layers[-(i + 1)]

        for param in layer_to_unfreeze.parameters():
            param.requires_grad = True
    for param in model.classifier.parameters():
        param.requires_grad = True

    # verify
    for idx, feat in enumerate(model.features):
        if any(param.requires_grad for param in feat.parameters()):
            print(f"layer {idx} requires grad")
    if all(param.requires_grad for param in model.classifier.parameters()):
        print("classifier requires grad")

In [30]:
partial_freeze(model, 3)  # type: ignore

layer 6 requires grad
layer 7 requires grad
layer 8 requires grad
classifier requires grad


In [31]:
optimizer = torch.optim.Adam(
    [
        {"params": model.features[6].parameters(), "lr": 1e-5},
        {"params": model.features[7].parameters(), "lr": 1e-5},
        {"params": model.features[8].parameters(), "lr": 1e-4},
        {"params": model.classifier.parameters(), "lr": 1e-3},
    ]
)

In [32]:
num_epochs = 20
model, metrics = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_function=loss_function,
    optimizer=optimizer,
    scheduler=None,
    num_epochs=num_epochs,
    device=device,
)

--- Training Started ---
Epoch [1/20], Train Loss: 57.7327, Val Loss: 46.8348, Val Accuracy: 86.81%
Epoch [2/20], Train Loss: 32.1807, Val Loss: 33.8008, Val Accuracy: 90.07%
Epoch [3/20], Train Loss: 22.8592, Val Loss: 26.9050, Val Accuracy: 91.53%
Epoch [4/20], Train Loss: 16.7708, Val Loss: 23.1395, Val Accuracy: 92.35%
Epoch [5/20], Train Loss: 13.6837, Val Loss: 20.7668, Val Accuracy: 92.83%
Epoch [6/20], Train Loss: 10.9851, Val Loss: 19.0023, Val Accuracy: 93.65%
Epoch [7/20], Train Loss: 9.2025, Val Loss: 18.3317, Val Accuracy: 93.57%
Epoch [8/20], Train Loss: 8.1604, Val Loss: 16.5128, Val Accuracy: 93.97%
Epoch [9/20], Train Loss: 7.0451, Val Loss: 16.3771, Val Accuracy: 94.14%
Epoch [10/20], Train Loss: 5.8157, Val Loss: 15.0439, Val Accuracy: 94.71%
Epoch [11/20], Train Loss: 5.4354, Val Loss: 15.1333, Val Accuracy: 95.03%
Epoch [12/20], Train Loss: 4.8763, Val Loss: 14.6380, Val Accuracy: 94.63%
Epoch [13/20], Train Loss: 4.3666, Val Loss: 14.3380, Val Accuracy: 94.95%
Epo

In [33]:
test_loss, test_accuracy = val_epoch(model, test_loader, loss_function, device)
print(f"{test_loss=}\n{test_accuracy=}\n\nTrain Metrics:{metrics}")

test_loss=11.223345762258395
test_accuracy=95.43973941368078

Train Metrics:[[57.73268048365911, 32.180651072329944, 22.859155382712682, 16.770786400636037, 13.683650664819611, 10.985098039607207, 9.202515818840928, 8.160373739070362, 7.0450674932036135, 5.815664710435603, 5.435371993896034, 4.876349952816963, 4.366569487419393, 4.150243783411052, 3.9002724106320077, 3.3981987554579973, 2.878406037224664, 2.8515871602214045, 2.459335545533233, 2.385151454889112], [46.83478626012802, 33.800794154405594, 26.90496068894863, 23.139497050642966, 20.76676230430603, 19.002262610197068, 18.33174939453602, 16.512799967825412, 16.3770557269454, 15.043881210684777, 15.133332519233226, 14.638042738288641, 14.337961923331022, 13.941885854303838, 14.43545476347208, 14.008383994549513, 13.519754317402839, 12.912141873687506, 12.943714026361704, 12.716822949051856], [86.80781758957654, 90.06514657980456, 91.53094462540717, 92.34527687296416, 92.83387622149837, 93.64820846905538, 93.56677524429968, 93.

In [36]:
weight_dir = BASE_DIR / "model_weights"
torch.save(model.state_dict(), weight_dir / "ft_EfficientNet-B0.pth")